# MMSD: pretrained ResNet-50 V2 image features


In [1]:
%pip install timm numpy Pillow tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import platform
import shutil
import time
from pathlib import Path

import numpy as np
import PIL
from PIL import Image
import torch
import torchvision
import timm
from tqdm.auto import tqdm

PART = 3
NUM_PARTS = 3
DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-dataset')
IMAGE_DIR = DATA_ROOT / 'raw/dataset_image'
OUTPUT_BASE = Path('/kaggle/working/features/resnetv2_50_a1h_in1k')
OUTPUT_DIR = OUTPUT_BASE / f'part_{PART}'
assert 1 <= PART <= NUM_PARTS
BATCH_SIZE = 32
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, 'Device:', DEVICE, 'GPUs:', torch.cuda.device_count())


PyTorch: 2.11.0+cu130 Device: cuda GPUs: 2


In [3]:
all_tweet_ids = sorted(path.stem for path in IMAGE_DIR.glob('*.jpg'))
tweet_ids = [str(i) for i in np.array_split(all_tweet_ids, NUM_PARTS)[PART - 1]]
print('Total:', len(all_tweet_ids), 'This part:', PART, len(tweet_ids))
print('Feature storage:', len(tweet_ids) * 196 * 2048 * 4 / 1e9, 'GB plus headers')


Total: 24635 This part: 3 8211
Feature storage: 13.183844352 GB plus headers


In [4]:
MODEL_NAME = 'resnetv2_50.a1h_in1k'
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0, global_pool='')
model.requires_grad_(False).eval().to(DEVICE)
mean = torch.tensor(model.pretrained_cfg['mean'], dtype=torch.float32).view(1, 3, 1, 1)
std = torch.tensor(model.pretrained_cfg['std'], dtype=torch.float32).view(1, 3, 1, 1)

class SpatialEncoder(torch.nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, inputs):
        return self.backbone.forward_features(inputs)

encoder = SpatialEncoder(model).eval()
if torch.cuda.device_count() > 1:
    encoder = torch.nn.DataParallel(encoder)

@torch.inference_mode()
def extract_batch(ids):
    images = []
    for tweet_id in ids:
        with Image.open(IMAGE_DIR / f'{tweet_id}.jpg') as image:
            image = image.convert('RGB').resize((448, 448), Image.Resampling.BICUBIC)
            images.append(np.asarray(image, dtype=np.float32))
    inputs = torch.from_numpy(np.stack(images)).permute(0, 3, 1, 2) / 255.0
    inputs = ((inputs - mean) / std).to(DEVICE)
    spatial = encoder(inputs)
    assert tuple(spatial.shape) == (len(ids), 2048, 14, 14), spatial.shape
    features = spatial.flatten(2).transpose(1, 2).contiguous().cpu().numpy()
    assert features.dtype == np.float32 and np.isfinite(features).all()
    return features


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

In [5]:
manifest = {
    'upstream_revision': 'f42b16510208624d91fa545ca9bb64c6335f971e',
    'encoder': MODEL_NAME, 'weights': 'timm ImageNet-1k a1h',
    'feature_layer': 'forward_features (after final normalization)', 'training': False,
    'resize': [448, 448], 'color': 'RGB', 'resampling': 'Pillow BICUBIC',
    'preprocessing': 'direct resize without cropping; RGB / 255, then channel normalization',
    'mean': list(model.pretrained_cfg['mean']), 'std': list(model.pretrained_cfg['std']),
    'output_shape': [196, 2048], 'dtype': 'float32',
    'flatten_order': 'row-major spatial grid',
    'part': PART, 'num_parts': NUM_PARTS, 'tweet_ids': tweet_ids,
    'torch': torch.__version__, 'torchvision': torchvision.__version__, 'timm': timm.__version__,
    'numpy': np.__version__, 'pillow': PIL.__version__,
    'python': platform.python_version(), 'seed': SEED,
    'deviation': 'timm a1h ImageNet-1k checkpoint and direct 448 resize; original checkpoint and fine-tuning unavailable',
}
print(json.dumps({k: v for k, v in manifest.items() if k != 'tweet_ids'}, indent=2))


{
  "upstream_revision": "f42b16510208624d91fa545ca9bb64c6335f971e",
  "encoder": "resnetv2_50.a1h_in1k",
  "weights": "timm ImageNet-1k a1h",
  "feature_layer": "forward_features (after final normalization)",
  "training": false,
  "resize": [
    448,
    448
  ],
  "color": "RGB",
  "resampling": "Pillow BICUBIC",
  "preprocessing": "direct resize without cropping; RGB / 255, then channel normalization",
  "mean": [
    0.5,
    0.5,
    0.5
  ],
  "std": [
    0.5,
    0.5,
    0.5
  ],
  "output_shape": [
    196,
    2048
  ],
  "dtype": "float32",
  "flatten_order": "row-major spatial grid",
  "part": 3,
  "num_parts": 3,
  "torch": "2.11.0+cu130",
  "torchvision": "0.26.0+cu130",
  "timm": "1.0.30",
  "numpy": "2.1.3",
  "pillow": "13.0.0",
  "python": "3.13.16",
  "seed": 42,
  "deviation": "timm a1h ImageNet-1k checkpoint and direct 448 resize; original checkpoint and fine-tuning unavailable"
}


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_DIR / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Cache mismatch: use a new output directory.'
else:
    assert not list(OUTPUT_DIR.glob('*.npy')), 'Existing features have no provenance; use a new directory.'
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
pending = []
for tweet_id in tweet_ids:
    path = OUTPUT_DIR / f'{tweet_id}.npy'
    if path.exists():
        cached = np.load(path, allow_pickle=False, mmap_mode='r')
        assert cached.shape == (196, 2048) and cached.dtype == np.float32, path
        assert np.isfinite(cached).all(), path
        del cached
    else:
        pending.append(tweet_id)
required = len(pending) * (196 * 2048 * 4 + 128) + 1024**3
assert shutil.disk_usage(OUTPUT_DIR).free >= required, 'Insufficient free disk space (including 1 GiB margin).'


In [7]:
start = time.perf_counter()
progress = tqdm(total=len(pending), desc=f'Part {PART}', unit='image')
for offset in range(0, len(pending), BATCH_SIZE):
    ids = pending[offset:offset + BATCH_SIZE]
    batch = extract_batch(ids)
    for tweet_id, feature in zip(ids, batch):
        target = OUTPUT_DIR / f'{tweet_id}.npy'
        temporary = target.with_suffix('.npy.tmp')
        with temporary.open('wb') as stream:
            np.save(stream, feature, allow_pickle=False)
        os.replace(temporary, target)
    progress.update(len(ids))
progress.close()
assert all((OUTPUT_DIR / f'{i}.npy').is_file() for i in tweet_ids)
print('Complete:', len(tweet_ids), 'feature files in', OUTPUT_DIR)
print('Elapsed extraction time for this run:', time.perf_counter() - start, 'seconds')


Part 3:   0%|          | 0/8211 [00:00<?, ?image/s]

Complete: 8211 feature files in /kaggle/working/features/resnetv2_50_a1h_in1k/part_3
Elapsed extraction time for this run: 981.8428966690001 seconds


In [8]:
import shutil

shutil.make_archive("/kaggle/working/output", "zip", "/kaggle/working")

'/kaggle/working/output.zip'